# ПЗ 1 — Обработка изображений (OpenCV)

Загружаем изображение, применяем базовые преобразования OpenCV, читаем числовой текст через EasyOCR.

In [ ]:
!pip install opencv-python-headless easyocr pandas matplotlib -q

In [ ]:
from google.colab import files

uploaded = files.upload()
file_path = list(uploaded.keys())[0]

In [ ]:
import cv2
import numpy as np
import easyocr
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display

reader = easyocr.Reader(['ru', 'en'], gpu=True)

img = cv2.imread(file_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray    = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

blurred = cv2.GaussianBlur(gray, (5, 5), 0)

sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
sobel_combined = cv2.magnitude(sobelx, sobely)

_, mask = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

R, G, B = cv2.split(img_rgb)

In [ ]:
plt.figure(figsize=(20, 12))

panels = [
    (img_rgb,        'оригинал'),
    (gray,           'ч/б'),
    (blurred,        'сглаживание'),
    (sobel_combined, 'контуры'),
    (mask,           'маска'),
    (R,              'канал R'),
    (G,              'канал G'),
    (B,              'канал B'),
]

for i, (image, title) in enumerate(panels):
    plt.subplot(2, 4, i + 1)
    plt.imshow(image, cmap='gray' if len(image.shape) == 2 else None)
    plt.title(title)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
results = reader.readtext(gray)
rows = []

for bbox, text, prob in results:
    digits = ''.join(filter(str.isdigit, text))
    if len(digits) >= 5:
        rows.append({
            'изображение': file_path.split('/')[-1],
            'результат':   digits,
            'уверенность': round(prob, 3),
        })
        break

if rows:
    display(pd.DataFrame(rows))
else:
    print('числовой текст не найден')